In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

from pyspark.sql import functions as F
from nyc_taxi.config import PipelineConfig, ref_month_str
from nyc_taxi.bronze import write_partition
from nyc_taxi.silver import split_valid_rejected
from nyc_taxi.spark_utils import configure_session

cfg = PipelineConfig()
configure_session(spark)

In [0]:
for taxi_type in cfg.taxi_types:
    bronze = spark.table(cfg.table("bronze", f"{taxi_type}_tripdata"))
    for m in cfg.months():
        month_df = bronze.filter(F.col("ref_month") == ref_month_str(m))
        valid, rejected = split_valid_rejected(month_df, taxi_type, m)
        write_partition(valid, cfg.table("silver", f"{taxi_type}_tripdata"), m)
        write_partition(rejected, cfg.table("silver", f"{taxi_type}_tripdata_quarantine"), m)
        print(f"{taxi_type} · {ref_month_str(m)} processado")

In [0]:
for taxi_type in cfg.taxi_types:
    s = cfg.table("silver", f"{taxi_type}_tripdata")
    q = cfg.table("silver", f"{taxi_type}_tripdata_quarantine")
    display(spark.sql(f"""
      WITH v AS (SELECT ref_month, COUNT(*) AS valid FROM {s} GROUP BY ref_month),
           r AS (SELECT ref_month, reject_reason, COUNT(*) AS n FROM {q} GROUP BY ref_month, reject_reason)
      SELECT '{taxi_type}' AS taxi_type, v.ref_month, v.valid,
             COALESCE(SUM(r.n), 0) AS rejected,
             ROUND(100 * COALESCE(SUM(r.n), 0) / (v.valid + COALESCE(SUM(r.n), 0)), 3) AS pct_rejected,
             MAP_FROM_ENTRIES(COLLECT_LIST(CASE WHEN r.reject_reason IS NOT NULL
                                                THEN STRUCT(r.reject_reason, r.n) END)) AS by_reason
      FROM v LEFT JOIN r USING (ref_month)
      GROUP BY v.ref_month, v.valid
      ORDER BY v.ref_month"""))